# Chapter three: Latent Space Analysis

In [ ]:
## Generator Constraint: Prototype Regularization

Based on the latent space analysis above, we define the per-class centroids $\bar{z}_y$
that can serve as the anchor for prototype regularization.

The modified generator loss with constraint is:

$$J(w) := \underbrace{\mathbb{E}_{y,z}\left[\ell\left(\sigma\left(\frac{1}{K}\sum_k g(z;\theta^p_k)\right), y\right)\right]}_{\text{ensemble classification loss}} + \underbrace{\lambda \cdot \mathbb{E}_{y, z\sim G_w(\cdot|y)} \left[ \|z - \bar{z}_y\|^2 \right]}_{\text{prototype regularization}}$$

This ensures generated representations stay near the latent region occupied by real
patients of each class. The hyperparameter $\lambda$ controls the trade-off between
ensemble consensus (original objective) and distributional fidelity (constraint).

The main idea is that since the statistics can not be trully trusted in order to make real clinical vectors, they must be generated close to real ones, like reducing the noise from it.

In [ ]:
if LATENT_LOADED:
    # ── PCA of latent space, colored by class ─────────────────────────────────
    pca = PCA(n_components=10)
    Z_pca = pca.fit_transform(Z)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Latent Space Geometry (z ∈ ℝ¹²⁸, penultimate layer)',
                 fontweight='bold', fontsize=13)

    # Subsample for visualization
    idx_sub = np.random.default_rng(SEED).choice(len(Z), size=min(8000, len(Z)), replace=False)
    Z_sub   = Z_pca[idx_sub]
    y_sub   = y[idx_sub]

    # PC1 vs PC2
    for label, (name, color) in zip([0, 1], PALETTE.items()):
        mask = y_sub == label
        axes[0].scatter(Z_sub[mask, 0], Z_sub[mask, 1],
                        c=color, alpha=0.3, s=5, label=name, rasterized=True)
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    axes[0].set_title('PCA: PC1 vs PC2')
    axes[0].legend(markerscale=3, fontsize=9)

    # Explained variance
    axes[1].bar(range(1, 11), pca.explained_variance_ratio_ * 100,
                color='steelblue', alpha=0.8, edgecolor='white')
    axes[1].plot(range(1, 11), np.cumsum(pca.explained_variance_ratio_) * 100,
                 'o-', color='darkorange', linewidth=2, markersize=6, label='Cumulative')
    axes[1].set_xlabel('Principal component')
    axes[1].set_ylabel('% variance explained')
    axes[1].set_title('Latent space: variance explained')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('figures/09_latent_space_pca.png', bbox_inches='tight')
    plt.show()

    # ── Per-class statistics in latent space ──────────────────────────────────
    Z0 = Z[y == 0]
    Z1 = Z[y == 1]

    print('\nLatent space class-conditional statistics:')
    print(f'  Class 0 — mean norm: {np.linalg.norm(Z0.mean(0)):.3f} | '
          f'mean intra-class dist (sample): {np.mean(np.linalg.norm(Z0[:500] - Z0[:500].mean(0), axis=1)):.3f}')
    print(f'  Class 1 — mean norm: {np.linalg.norm(Z1.mean(0)):.3f} | '
          f'mean intra-class dist (sample): {np.mean(np.linalg.norm(Z1[:500] - Z1[:500].mean(0), axis=1)):.3f}')
    inter_dist = np.linalg.norm(Z0.mean(0) - Z1.mean(0))
    print(f'  Inter-class centroid distance: {inter_dist:.3f}')
    print(f'  Latent dimension: {Z.shape[1]}')
    print(f'\n  → These are the prototype centroids z̄_y that a constrained generator should stay near.')

In [ ]:
if LATENT_LOADED:
    # Compute and save per-class prototypes
    prototypes = {
        0: Z[y == 0].mean(axis=0),  # z̄_{y=0}
        1: Z[y == 1].mean(axis=0),  # z̄_{y=1}
    }
    prototype_stds = {
        0: Z[y == 0].std(axis=0),
        1: Z[y == 1].std(axis=0),
    }

    print('Prototype centroids (z̄_y) computed:')
    for cls in [0, 1]:
        print(f'  z̄_{{y={cls}}}: shape={prototypes[cls].shape}, '
              f'norm={np.linalg.norm(prototypes[cls]):.3f}, '
              f'mean std={prototype_stds[cls].mean():.3f}')

    # Suggested λ: a fraction of the typical intra-class variance
    # so the constraint is soft but meaningful
    intra_var_0 = np.mean(np.var(Z[y == 0], axis=0))
    intra_var_1 = np.mean(np.var(Z[y == 1], axis=0))
    suggested_lambda_scale = 0.1  # 10% of intra-class variance
    print(f'\nMean intra-class variance: class0={intra_var_0:.3f}, class1={intra_var_1:.3f}')
    print(f'Suggested λ starting point: ~{suggested_lambda_scale} '
          f'(ablate over [0.0, 0.01, 0.1, 1.0])')

    # Visualize prototype + 1-std ellipse in PCA space
    fig, ax = plt.subplots(figsize=(7, 6))
    for label, (name, color) in zip([0, 1], PALETTE.items()):
        mask = y_sub == label
        ax.scatter(Z_sub[mask, 0], Z_sub[mask, 1],
                   c=color, alpha=0.2, s=5, rasterized=True)
        # Project prototype into PCA space
        proto_pca = pca.transform(prototypes[label].reshape(1, -1))[0]
        ax.scatter(*proto_pca[:2], c=color, s=120, marker='*',
                   edgecolors='black', linewidth=1, zorder=5,
                   label=f'{name} (prototype z̄_y)')

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('Latent space: class prototypes z̄_y\n'
                 '(generator constraint anchors)', fontweight='bold')
    ax.legend(markerscale=2, fontsize=9)
    plt.tight_layout()
    plt.savefig('figures/10_latent_prototypes.png', bbox_inches='tight')
    plt.show()

    np.save('prototypes_class0.npy', prototypes[0])
    np.save('prototypes_class1.npy', prototypes[1])
    print('\nPrototypes saved to prototypes_class0.npy / prototypes_class1.npy')
else:
    print('Latent analysis skipped — train centralized model first.')

## Latent Space Analysis: Where the Generator Actually Operates

With `hidden_dim = 512`, your MLP architecture is:

```
Feature extractor θ^f:  99 → 512 → 256 → 128   (LayerNorm + ReLU + Dropout)
Predictor head θ^p:      128 → 2
```

The FedGen generator operates in **Z = ℝ¹²⁸** (the 128-dimensional penultimate layer).
Here we load the saved centralized MLP and inspect the class-conditional geometry of
the latent space — this is what the generator is trying to approximate.

If the two classes are well-separated in latent space, the generator has a clearly
defined target distribution. If they overlap significantly, the generator's task is
harder and the inductive bias is weaker.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=2, dropout=0.3):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.LayerNorm(hidden_dim // 4), nn.ReLU(), nn.Dropout(dropout),
        )
        self.predictor = nn.Linear(hidden_dim // 4, output_dim)

    def forward(self, x):
        return self.predictor(self.feature_extractor(x))

    def encode(self, x):
        return self.feature_extractor(x)


# ── Load centralized checkpoint ───────────────────────────────────────────────
CKPT_PATH = '../01_Centralized/models/mlp_centralized.pt'
print(f'\nAttempting to load centralized checkpoint from: {os.path.exists(CKPT_PATH)}')
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    hidden_dim = ckpt.get('hidden_dim', 256)
    dropout    = ckpt.get('dropout',    0.3)
    scaler     = ckpt.get('scaler',     StandardScaler())

    model = MLP(input_dim=X.shape[1], hidden_dim=hidden_dim, dropout=dropout).to(device)

    # Remap state_dict keys: original model uses 'net.*', new uses 'feature_extractor.*' / 'predictor.*'
    orig_sd = ckpt['model_state_dict']
    new_sd  = {}
    fe_layers = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # indices in original net.Sequential
    # Map net.0..net.11 → feature_extractor, net.12 → predictor
    for k, v in orig_sd.items():
        if k.startswith('net.'):
            idx = int(k.split('.')[1])
            rest = '.'.join(k.split('.')[2:])
            if idx < 12:  # feature extractor layers
                new_sd[f'feature_extractor.{idx}.{rest}'] = v
            else:
                new_sd[f'predictor.{rest}'] = v
        else:
            new_sd[k] = v

    model.load_state_dict(new_sd, strict=False)
    model.eval()

    # ── Extract latent representations ────────────────────────────────────────
    X_scaled = scaler.transform(X) if hasattr(scaler, 'mean_') else X
    X_t = torch.tensor(X_scaled, dtype=torch.float32)

    latent_vecs = []
    with torch.no_grad():
        loader = DataLoader(TensorDataset(X_t), batch_size=2048)
        for (xb,) in loader:
            latent_vecs.append(model.encode(xb.to(device)).cpu().numpy())
    Z = np.concatenate(latent_vecs, axis=0)

    print(f'Latent space shape: {Z.shape}  (n_samples × latent_dim)')
    LATENT_LOADED = True
else:
    print(f'Checkpoint not found at {CKPT_PATH}.')
    print('Skipping latent space analysis. Run this after training the centralized model.')
    LATENT_LOADED = False